
# 🧭 Day 3 — Pipelines, Modularity & Debugging with DSPY

<a href="https://colab.research.google.com/github/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/lectures/03-pipeline_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

*How AI engineers move from single prompts to reliable, modular LLM systems.*

---

## 🎯 Learning Objectives

By the end of this week, you should be able to:
1. Explain what a pipeline is and why modularity matters for reliability.  
2. Distinguish between monolithic and multi-step prompt designs.  
3. Configure and run a basic LLM in `dspy` using the OpenAI API.  
4. Build, inspect, and debug simple pipelines step-by-step.  
5. Use `dspy.inspect()` and logging to understand what’s actually happening inside an LLM pipeline.


In [ ]:
# @title Setup (Run this first)
!git clone --depth 1 -q https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials.git
import sys, platform
sys.path.append('./course-materials')
from course_utils import lab3_setup, show_mermaid

lab3_setup()
print(f"✅ Environment ready!")


## Crazy things can happen when it's hot (high temperatures)

In [ ]:
from course_utils import lab2_generate_samples
results = lab2_generate_samples(prompt="Describe a sunrise in one sentence.",
                                temperatures=[2],
                                n_per_temp=5)
results

In [ ]:
# @title limiting output with max_output_tokens
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="Describe a sunrise in one sentence.",
    temperature=2,
    max_output_tokens=20
)
display(response.output_text)

# 📅 From Prompts to Pipelines


## Section 1 — Lab L2 Recap: What Changed When Temperature Increased?

**Guiding Question:**  
> What did you notice when temperature increased? Was the model more creative, or less reliable?

Discuss:
- Higher temperature → more diverse completions.
- Lower temperature → more deterministic and stable outputs.



## Section 2 — Motivating Example: Why Modularity?

**Question:**  
> Why can’t we just write one giant prompt and call it a day?

We'll explore examples of LLM success and failure:
- **Success:** multi-step reasoning task (summarize → critique → rewrite).
- **Failure:** same task with one unstructured prompt.

**Reflection:**  
> What might have gone wrong?  
> How could you design the system to isolate each step?



## Section 3 — Concept: What Is a Pipeline?

A **pipeline** is a sequence of processing steps that move data forward.  
Each stage transforms or filters information before handing it off.

**Mathematical intuition:**

$$
P(Y|X) = \prod_{i=1}^{n} P(Y_i | Y_{<i}, X)
$$



In [8]:
# @title summarization pipeline
show_mermaid("""graph TD
  I["Input"] --> E["Extract"]
  E --> S["Simplify"]
  S --> Z["Summarize"]
  Z --> O["Output"]
""")

In [9]:
# @title Updated LLM Diagram
show_mermaid("""
graph TD
    %% === USER INPUT ===
    subgraph User Interaction
    U["👤 Users<br/>Queries / Inputs"]:::user --> IH["Input Handling<br/>• Formatting<br/>• Validation<br/>• Safety Filters"]:::process
    end

    %% === PIPELINE STRUCTURE ===
    subgraph Pipeline Modules
    IH --> M1["Module 1: Extraction<br/>• Pull key facts or entities"]:::module
    M1 --> M2["Module 2: Reasoning<br/>• Infer relationships<br/>• Compute answers"]:::module
    M2 --> M3["Module 3: Summarization<br/>• Convert structured data<br/>• Produce final text output"]:::module
    end

    %% === MODEL & DATA CONNECTIONS ===
    subgraph Core LLMs
    M1 --> L1["LLM A<br/>Specialized extractor"]:::model
    M2 --> L2["LLM B<br/>Reasoner / Planner"]:::model
    M3 --> L3["LLM C<br/>Writer / Stylist"]:::model
    end

    subgraph Model Training
    D["📚 Training Data Distribution<br/>• Text corpus<br/>• Domain knowledge<br/>• Bias sources"]:::data --> L1
    D --> L2
    D --> L3
    end

    %% === OUTPUT & MONITORING ===
    subgraph Output & Monitoring
    M3 --> OP["Output Processing<br/>• Formatting<br/>• Citations<br/>• Guardrails"]:::output
    OP --> O["🟢 Final Output"]:::output
    O --> LM["Logging & Monitoring<br/>• Per-module metrics<br/>• Failure localization<br/>• Drift detection"]:::monitor
    end

    %% === STYLES ===
    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef module fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef data fill:#fde2e4,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
    classDef monitor fill:#fefefe,stroke:#333,stroke-width:1px;
""")



## Section 4 — Mini DSPY Preview: The Simplest Predictor

**Guiding Question:**  
> What does it mean to describe a prompt as a *function*?


In [10]:
# @title configure dspy to use openai
import dspy
import os
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)

In [ ]:
# @title simple dspy function
predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")
print(result.answer)


In [ ]:
# @title inspect prompt
dspy.inspect_history()


## Section 5 — Debugging by Decomposition

**Guiding Question:**  
> What happens when a single step in a multi-step system fails?


In [ ]:
# @title sample failures
import random

steps = ["extract", "simplify", "summarize"]
for s in steps:
    success = random.random() > 0.3
    print(f"{s}: {'✅ success' if success else '❌ failure'}")



## Section 6 — Activity: Sketch a Real AI Pipeline

In pairs, pick an AI system (e.g., ChatGPT, Grammarly, Copilot).  
Sketch its internal pipeline (3–4 boxes max):  
*Input Handling → Intent Detection → LLM → Postprocessing*



## Section 7 — 5-Minute Concept Quiz

1. Why is modularity useful in AI pipelines?  
2. What is “failure localization”?  
3. Why might debugging be harder in a single giant prompt?  
4. How does `dspy` help with modularity?  
5. What does `dspy.inspect()` show?



## Section 8 — Unifying Diagram v2



In [14]:
# @title pipeline

show_mermaid(
"""
graph TD
  U["User Input"] --> IH["Input Handling"]
  IH --> S1["Step 1: Extract Info"]
  S1 --> S2["Step 2: Simplify"]
  S2 --> LLM["Core LLM"]
  LLM --> OP["Output Processing"]
  OP --> M["Monitoring"]
"""
)


<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 1)</b></summary>

- Timing: 10 + 10 + 15 + 15 + 15 + 5 + 5  
- Use `dspy.inspect()` live to reveal prompts.  
- Encourage students to draw diagrams and share debugging analogies.  
- Wrap with a quiz and transition: “Next class, we’ll *build* pipelines in DSPY.”

</details>


# 📅 Building and Debugging Pipelines with DSPY


## Section 2 — Typed Predictors


In [ ]:
# @title adding types
sentiment = dspy.Predict("sentence -> sentiment: bool")
result = sentiment(sentence="The sky is blue.")
print("Sentiment:", result.sentiment)
type(result.sentiment)

In [ ]:
dspy.inspect_history()

In [ ]:
# @title multiple outputs
qa = dspy.Predict("question -> reasoning, answer")
resp = qa(question="Why do leaves change color in autumn?")
print("Reasoning:", resp.reasoning)
print("Answer:", resp.answer)


In [ ]:
dspy.inspect_history()

In [ ]:
# @title summarization pipeline
extract = dspy.Predict("article -> key_points")
simplify = dspy.Predict("key_points -> summary")
tag = dspy.Predict("summary -> tags: list[str]")

def summarize_pipeline(article):
    key_points = extract(article=article).key_points
    summary = simplify(key_points=key_points).summary
    tags = tag(summary=summary).tags
    return summary, tags

article = "Photosynthesis converts light energy into chemical energy..."
display(summarize_pipeline(article))


In [ ]:
dspy.inspect_history()

In [ ]:
# @title Using try/except blocks to safely execute pipeline steps
def safe_run(predictor, **kwargs):
    try:
        result = predictor(**kwargs)
        if not result:
            raise ValueError("Empty output")
        return result
    except Exception as e:
        print(f"⚠️ Error in {predictor}: {e}")
        return None

result = safe_run(extract, text=[])
result


In [ ]:
# @title sample trace
import pandas as pd

trace = [
    {"step": "extract", "status": "✅", "details": "3 key points"},
    {"step": "simplify", "status": "✅", "details": "Readable summary"},
    {"step": "tag", "status": "⚠️", "details": "Low confidence tags"}
]

pd.DataFrame(trace)


In [ ]:
# @title Using Classes with dspy.Module
# here, we wrap our pipeline in a class called SummarizerModule
# In the constructor (__init__), we create three dspy Predict objects
# for the three stages of our pipeline.

# Then, we implement the forward function, which runs the pipeline
# and returns the results.
class SummarizerModule(dspy.Module):
    def __init__(self):
      self.extract = dspy.Predict("article -> key_points")
      self.simplify = dspy.Predict("key_points -> summary")
      self.tag = dspy.Predict("summary -> tags: list[str]")

    def forward(self, article):
      key_points = self.extract(article=article).key_points
      summary = self.simplify(key_points=key_points).summary
      tags = self.tag(summary=summary).tags
      return summary, tags

summarizer = SummarizerModule()
summary, tags = summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive")
print('summary=', summary)
print('tags=', tags)

In [ ]:
# @title Customizing instructions in dspy using the Signature class.
# We add an additional parameter `instructions`.
spanish_summarizer = dspy.Predict(
                        dspy.Signature("article -> summary",
                        instructions='Summarize the article in Spanish'))

print(spanish_summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive"))
# dspy.inspect_history(1)

In [ ]:
#@title Same thing, but defining fields explicitly as class attributes.
class SpanishSummarizer(dspy.Signature):
  """
  Summarize the article in Spanish
  """
  article: str = dspy.InputField()
  summary: str = dspy.OutputField()

spanish_summarizer2 = dspy.Predict(SpanishSummarizer)
print(spanish_summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive"))


In [ ]:
# @title logging steps
import pandas as pd
pd.set_option("display.max_colwidth", None)  # don't truncate/wrap long text

logs = []
article = "AI systems are more interpretable when modular."
summary, tags = summarizer(article)
logs.append({"input": article, "output": summary})

pd.DataFrame(logs)


In [ ]:
# @title Add logging to summarizer.
class SummarizerModule(dspy.Module):
    def __init__(self):
      self.extract = dspy.Predict("article -> key_points")
      self.simplify = dspy.Predict("key_points -> summary")
      self.tag = dspy.Predict("summary -> tags: list[str]")

    def forward(self, article):
      key_points = self.extract(article=article).key_points
      summary = self.simplify(key_points=key_points).summary
      tags = self.tag(summary=summary).tags
      return summary, key_points, tags

logs = []
summarizer = SummarizerModule()
summary, key_points, tags = summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive")
logs.append({'article': article,
             'key_points': key_points,
             'tags': tags,
             'summary': summary})

pd.DataFrame(logs)


In [ ]:
# @title Example loop for many inputs, logging each one.
articles = [
    "The sun provides energy for plants. Plants use photosynthesis to convert sun into energy.Plants need energy to survive",
    "AI systems are more interpretable when modular. Modular code makes things easier to debug, and also allows you to reuse code in other parts of your system."
]

summarizer = SummarizerModule()
logs = []
for article in articles:
  summary, key_points, tags = summarizer(article=article)
  logs.append({'article': article,
              'key_points': key_points,
              'tags': tags,
              'summary': summary})
pd.DataFrame(logs)

## Group Activity

Let's do a group activity to practice breaking problems into pipelines of modular components.

E.g., for our Summarizer, we broke down the problem of

> "Summarize an article"

into three components:

1. **Extract key points:**
    - input: article (str). Article to be summarized
    - output: key_points (str). A string listing key points in the article.
2. **Simplify key points:**
    - input: key_points
    - output: summary (str). A summary of the key points.
3. **Create tags:**
    - input: summary (str)
    - output: tags (list[str]). A list of tags related to the summary.


### Instructions

Please break into groups of 2-4 and work through the sheet together for a differnt problem.

You can pick from here or makeup your own.

1. **Campus Info Assistant**: chatbot to answer student questions.

2. **Study assistant**: help users study for an exam

3. **Resume optimizer**: Given a resume and a job posting, help the user customize their resume for the job posting.


<br><br>

**Tips:**
- Name the fields. Don’t pass ‘everything’ forward.

- One box = one job. If it’s doing two jobs, split it.

- What modules are most likely to fail and why?



<details>
<summary>🧑‍🏫 <b>Instructor Notes</b></summary>

- Timing: 10 + 10 + 15 + 20 + 10 + 5 + 5  
- Have students run `dspy.inspect()` and `safe_run()` interactively.  
- Encourage experimentation: intentionally break steps and observe errors.  
- Close by linking debugging practices to system reliability.

</details>
